In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import time
import anndata as an
import scanpy as sc
import scanpy.external as sce
import h5py

In [2]:
%%time
path = "/nfs/turbo/umms-indikar/shared/projects/hybrid_reprogramming/anndata/processed_all_groups_v2.h5ad"
adata = sc.read_h5ad(path)
sc.logging.print_memory_usage()
adata

Memory usage: current 2.40 GB, difference +2.40 GB
CPU times: user 285 ms, sys: 1.31 s, total: 1.6 s
Wall time: 5.39 s


/nfs/turbo/umms-indikar/Jillian/conda-envs/rapids/lib/python3.13/site-packages/anndata/logging.py:57: FutureWarning: The specified parameters ('newline',) are no longer positional. Please specify them like `newline=False`
  print(format_memory_usage(get_memory_usage(), msg, newline))


AnnData object with n_obs × n_vars = 16296 × 25126
    obs: 'MYOD-fb_counts', 'PRRX1-fb_counts', 'PRRX1_MYOD-fb_counts', 'assigned_condition', 'total_fb_counts', 'condition_counts_rate', 'G1-fb_counts', 'G2M-fb_counts', 'S-fb_counts', 'dataset', 'total_reads', 'total_genes', 'pooled_condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', 'mt_outlier', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden', 'cluster_str', 'leiden_split'
    var: 'gene_id', 'gene_type', 'Chromosome', 'Start', 'End', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'mean', 'st

In [9]:
# # expr matrix

# adata.X = adata.layers['log_norm'].copy()

# gene_names = adata.var_names

# # dense format
# X = adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray()

# X_df = pd.DataFrame(
#     X,
#     index=adata.obs_names,
#     columns=adata.var_names,
# ).T

# print(X_df.shape)
# X_df.head()

(25126, 16296)


,AAACCAAAGCAACTGC_hybrid,AAACCAAAGTAGGGCA_hybrid,AAACCAAAGTCTAGGC_hybrid,AAACCATTCACGTAAT_hybrid,AAACCATTCAGGCAGA_hybrid,AAACCATTCCGTAATG_hybrid,AAACCCGCACCATAGG_hybrid,AAACCCGCATCCAATT_hybrid,AAACCCTGTAACGACA_hybrid,AAACCCTGTCAATGTC_hybrid,...,TGTGCTGGTTCCGTGA_control,TGTGCTGGTTGCTTAA_control,TGTGCTGGTTGGACGG_control,TGTGGTCAGCCCTTCA_control,TGTGGTCAGCTAACTG_control,TGTGGTCAGTGAGCCT_control,TGTGGTTGTAGCCCTA_control,TGTGGTTGTTGAAGGC_control,TGTGTTAGTTGGTAAG_control,TGTGTTGAGCCTATCT_control
gene_name,,,,,,,,,,,,,,,,,,,,,
A1BG,0.0,0.0,0.0,0.709331,0.433809,0.0,0.0,0.481149,0.449259,1.33782,...,1.430147,0.367738,1.033693,0.0,1.091004,0.000000,0.0,0.0,0.0,0.802924
A1BG-AS1,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.449259,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.771962,0.0,0.0,0.0,0.000000
A1CF,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
A2M,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000
A2M-AS1,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000


In [4]:
# phenoData
cols_to_keep = [
    'dataset', 'total_reads', 'total_genes', 'pooled_condition', 'phase', 'leiden_split',
]

pdf = adata.obs[cols_to_keep].copy()
print(pdf.shape)
pdf.head()

(16296, 6)


,dataset,total_reads,total_genes,pooled_condition,phase,leiden_split
AAACCAAAGCAACTGC_hybrid,Hybrid,18815.0,4890,siPRRX1/mmMYOD1,G2M,H1
AAACCAAAGTAGGGCA_hybrid,Hybrid,9679.0,3584,mmMYOD1,S,M1
AAACCAAAGTCTAGGC_hybrid,Hybrid,28392.0,6359,siPRRX1/mmMYOD1,G1,H1
AAACCATTCACGTAAT_hybrid,Hybrid,9684.0,3369,siPRRX1,G2M,P1
AAACCATTCAGGCAGA_hybrid,Hybrid,18413.0,4453,siPRRX1,G2M,P1


In [6]:
# featureData
cols_to_keep = [
    'gene_id', 'gene_type', 'Chromosome', 'Start', 'End',
]


fdf = adata.var[cols_to_keep].copy()
print(fdf.shape)

fdf.head()

(25126, 5)


,gene_id,gene_type,Chromosome,Start,End
gene_name,,,,,
A1BG,ENSG00000121410,protein_coding,chr19,58345177,58353492
A1BG-AS1,ENSG00000268895,lncRNA,chr19,58347717,58355455
A1CF,ENSG00000148584,protein_coding,chr10,50799408,50885675
A2M,ENSG00000175899,protein_coding,chr12,9067663,9116229
A2M-AS1,ENSG00000245105,lncRNA,chr12,9065162,9068689


In [11]:
outpath = "/scratch/indikar_root/indikar1/jrcwycy/HYB/monocle"

pdf.to_csv(f'{outpath}/cell_info.csv')
fdf.to_csv(f'{outpath}/gene_info.csv')